In [9]:
import json
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from utils import load_data, create_embedding_matrix, preprocess_text, to_padding, pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [10]:
# nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer =  SnowballStemmer('english')

train_claims_data = load_data('../data/train-claims.json')
evidence_data = load_data('../data/evidence.json')
dev_claims_data = load_data('../data/dev-claims.json')
# evidence_map = load_data('../data/curated/preprocessed_evidence_map.json')  
evidence_map = load_data('../data/curated/mild_nostopwords_evidence.json')  
filtered_evidence_map = load_data('../data/curated/mild_nostopwords_filtered_evidence.json')

In [13]:
label_mapping = {
    "SUPPORTS": 1,
    "REFUTES": -1,
    # "DISPUTED": 0,
    "NOT_ENOUGH_INFO": 0
}

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_entities = extract_entities(claim_text)
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    if claim_label == "DISPUTED":
            continue
    for eid in eids:
        evidence_text = evidence_map[eid]
        evidence_entities =
        data_for_dataframe.append({
                'claim_id': claim_id,
                # 'claim': preprocess_text(claim_text, stemmer=stemmer, stop_words=None),
                'claim_text': claim_text,
                'claim_entities': claim_entities,
                'evidence': evidence_text,
                'label': label_mapping[claim_label]
            })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

# train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
#     lambda x: ' '.join([evidence_map[evidence_id] for evidence_id in x])
# )

train_claims_df

,claim_id,claim,evidence,label
0,claim-126,el niño drove record high in global temperatur...,while climat chang can be due to natur forc or...,-1
1,claim-126,el niño drove record high in global temperatur...,this acceler is due most to human caus global ...,-1
2,claim-2510,in 1946 pdo switch to a cool phase,there is evid of revers in the prevail polar m...,1
3,claim-2510,in 1946 pdo switch to a cool phase,the pdo chang to a cool phase the pattern of t...,1
4,claim-2449,januari 2008 cap a 12 month period of global t...,with averag temperatur +8.1 47,0
...,...,...,...,...
3725,claim-502,but abnorm temperatur spike in februari and ea...,the coastlin see signific mild temperatur when...,0
3726,claim-3093,send oscil microwav from an antenna insid a va...,dielectr heat also known as electron heat radi...,1
3727,claim-3093,send oscil microwav from an antenna insid a va...,an exampl is absorpt or emiss of radio wave by...,1
3728,claim-3093,send oscil microwav from an antenna insid a va...,water fat and other substanc in the food absor...,1


In [26]:
import spacy

# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_lg")

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

data_for_dataframe = []
label_only = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_entities = extract_entities(claim_text)
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim_text': claim_text,
            'claim_entities': claim_entities,
            'evidence': eids
        })
    
# Create DataFrame
claim_entities_df = pd.DataFrame(data_for_dataframe)
claim_entities_df

,claim_id,claim_text,claim_entities,evidence
0,claim-752,[South Australia] has the most expensive elect...,"[(South Australia, LOC)]","[evidence-67732, evidence-572512]"
1,claim-375,when 3 per cent of total annual global emissio...,"[(3 per cent, MONEY), (Australia, GPE), (1.3 p...","[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,This means that the world is now 1C warmer tha...,"[(1C, CARDINAL)]","[evidence-889933, evidence-694262]"
3,claim-871,"“As it happens, Zika may also be a good model ...","[(Zika, PERSON), (second, ORDINAL)]","[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,Greenland has only lost a tiny fraction of its...,"[(Greenland, GPE)]","[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...,...
149,claim-2400,"'To suddenly label CO2 as a ""pollutant"" is a d...","[(Earth, LOC)]","[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,"after a natural orbitally driven warming, atmo...","[(800 years later, DATE)]","[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,Many of the world’s coral reefs are already ba...,[],"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,A recent study led by Lawrence Livermore Natio...,"[(Lawrence Livermore, PERSON), (National Labor...",[evidence-660755]


In [25]:
data_for_dataframe = []
for eid, evidence_text in evidence_data.items():
    data_for_dataframe.append({
        "eid": eid,
        "evidence_entities": extract_entities(evidence_text)
    })
evidence_entities = pd.DataFrame(data_for_dataframe)
evidence_entities

,eid,evidence_entities
0,evidence-0,"[(John Bennet Lawes, PERSON), (English, NORP)]"
1,evidence-1,"[(Lindberg, PERSON), (the age of 16, DATE), (N..."
2,evidence-2,"[(``Boston (Ladies of Cambridge, ORG), (Vampir..."
3,evidence-3,"[(Gerald Francis Goyer, PERSON), (October 20, ..."
4,evidence-4,"[(ECT, ORG)]"
...,...,...
1208822,evidence-1208822,[]
1208823,evidence-1208823,"[(Fyrde, PERSON)]"
1208824,evidence-1208824,"[(Dragon Storm, PERSON)]"
1208825,evidence-1208825,"[(Zeriuani, PERSON), (Slavs, NORP), (Zeriuani,..."


In [29]:
def match_evidence(claim_entities, evidence_entities_df, k=3):
    matched_evidences = []
    
    for _, row in evidence_entities_df.iterrows():
        evidence_id = row['eid']
        entities = row['evidence_entities']
        intersection = set([x for _,x in claim_entities]) & set([x for _, x in entities])
        matched_evidences.append((evidence_id, len(intersection)))

    # Sort the matched evidences by the length of the intersection (descending order)
    matched_evidences.sort(key=lambda x: x[1], reverse=True)

    # Return the top 3 evidence IDs with the most intersections
    top_evidence_ids = [evidence_id for evidence_id, _ in matched_evidences[:k]]
    return top_evidence_ids

claim_entities_df['top_evidence_ids'] = claim_entities_df['claim_entities'].apply(lambda x: match_evidence(x, evidence_entities))


In [30]:
claim_entities_df

,claim_id,claim_text,claim_entities,evidence,top_evidence_ids
0,claim-752,[South Australia] has the most expensive elect...,"[(South Australia, LOC)]","[evidence-67732, evidence-572512]","[evidence-5, evidence-35, evidence-74]"
1,claim-375,when 3 per cent of total annual global emissio...,"[(3 per cent, MONEY), (Australia, GPE), (1.3 p...","[evidence-996421, evidence-1080858, evidence-2...","[evidence-194, evidence-366, evidence-717]"
2,claim-1266,This means that the world is now 1C warmer tha...,"[(1C, CARDINAL)]","[evidence-889933, evidence-694262]","[evidence-3, evidence-20, evidence-27]"
3,claim-871,"“As it happens, Zika may also be a good model ...","[(Zika, PERSON), (second, ORDINAL)]","[evidence-422399, evidence-702226, evidence-28...","[evidence-38, evidence-103, evidence-185]"
4,claim-2164,Greenland has only lost a tiny fraction of its...,"[(Greenland, GPE)]","[evidence-52981, evidence-264761, evidence-947...","[evidence-1, evidence-6, evidence-9]"
...,...,...,...,...,...
149,claim-2400,"'To suddenly label CO2 as a ""pollutant"" is a d...","[(Earth, LOC)]","[evidence-409365, evidence-127519, evidence-85...","[evidence-5, evidence-35, evidence-74]"
150,claim-204,"after a natural orbitally driven warming, atmo...","[(800 years later, DATE)]","[evidence-368192, evidence-261690, evidence-20...","[evidence-1, evidence-3, evidence-5]"
151,claim-1426,Many of the world’s coral reefs are already ba...,[],"[evidence-1124018, evidence-995813, evidence-1...","[evidence-0, evidence-1, evidence-2]"
152,claim-698,A recent study led by Lawrence Livermore Natio...,"[(Lawrence Livermore, PERSON), (National Labor...",[evidence-660755],"[evidence-3, evidence-10, evidence-11]"
